In [1]:
import pandas as pd
import pickle

# تحديد مسار الملف المعالج (بصيغة pkl)
file_path = r'C:\Users\ev\Desktop\ANaconda\ir_project-main\processed_documents.pkl'

# قراءة الملف باستخدام مكتبة pickle
with open(file_path, 'rb') as f:
    df = pickle.load(f)

# عرض أول 5 أسطر للتأكد من نجاح العملية
print(df.head())

        doc_id                                      original_text  \
0  NCT00000102  Title: Congenital Adrenal Hyperplasia: Calcium...   
1  NCT00000104  Title: Does Lead Burden Alter Neuropsychologic...   
2  NCT00000105  Title: Vaccination With Tetanus and KLH to Ass...   
3  NCT00000106  Title: 41.8 Degree Centigrade Whole Body Hyper...   
4  NCT00000107  Title: Body Water Content in Cyanotic Congenit...   

                                        cleaned_text  
0  titl congenit adren hyperplasia calcium channe...  
1  titl lead burden alter neuropsycholog develop ...  
2  titl vaccin tetanu klh assess immun respons co...  
3  titl degre centigrad whole bodi hyperthermia t...  
4  titl bodi water content cyanot congenit heart ...  


In [2]:
import pandas as pd
import numpy as np

df = pickle.load(open(file_path, "rb"))

df = df.reset_index(drop=True)

df["doc_id"] = df["doc_id"].astype(str)
df["cleaned_text"] = df["cleaned_text"].astype(str)

In [3]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [doc.split() for doc in df["cleaned_text"]]

bm25 = BM25Okapi(tokenized_corpus)

In [5]:
queries = [
    "lead burden neuropsychology",
    "vaccination tetanus immune response",
    "hyperthermia body temperature study"
]

In [6]:
relevant_docs = {
    "lead burden neuropsychology": ["NCT00000104"],
    "vaccination tetanus immune response": ["NCT00000105"],
    "hyperthermia body temperature study": ["NCT00000106"]
}

In [7]:
X = []
y = []

def extract_features(query, doc_text, bm25_score):
    return [
        bm25_score,
        len(doc_text),
        len(query.split())
    ]

In [8]:
for query in queries:
    tokenized_query = query.split()
    
    scores = bm25.get_scores(tokenized_query)
    top_idx = np.argsort(scores)[::-1][:20]

    for idx in top_idx:
        doc_text = df.loc[idx, "cleaned_text"]
        doc_id = df.loc[idx, "doc_id"]

        features = extract_features(query, doc_text, scores[idx])
        X.append(features)

        y.append(1 if doc_id in relevant_docs[query] else 0)

In [9]:
import numpy as np

def bm25_search(query, top_k=20):

    tokenized_query = query.split()

    scores = bm25.get_scores(tokenized_query)

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in top_indices:
        results.append(
            (
                df.iloc[idx]["doc_id"],
                scores[idx]
            )
        )

    return results

In [10]:
print(bm25_search("lead burden neuropsychology")[:5])

[('NCT01573013', np.float64(11.27108002694473)), ('NCT00237952', np.float64(11.169904813604713)), ('NCT00227409', np.float64(10.766047405128074)), ('NCT01084694', np.float64(10.522496613464662)), ('NCT00014898', np.float64(9.933377817124242))]


In [11]:
from sentence_transformers import SentenceTransformer

bert_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [13]:
print(type(bert_model))

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


In [14]:
from sentence_transformers import util

def extract_ltr_features(query, doc_text, bm25_score, bert_model):

    query_emb = bert_model.encode(query, convert_to_tensor=True)
    doc_emb = bert_model.encode(doc_text, convert_to_tensor=True)

    bert_score = util.cos_sim(query_emb, doc_emb).item()

    doc_length = len(doc_text.split())

    return [
        bm25_score,
        bert_score,
        doc_length
    ]

In [15]:
X = []
y = []

for query in queries:

    tokenized_query = query.split()

    scores = bm25.get_scores(tokenized_query)

    top_indices = np.argsort(scores)[::-1][:20]

    for idx in top_indices:

        doc_id = df.iloc[idx]["doc_id"]
        doc_text = df.iloc[idx]["cleaned_text"]

        features = extract_ltr_features(
            query,
            doc_text,
            scores[idx],
            bert_model
        )

        X.append(features)

        y.append(
            1 if doc_id in relevant_docs[query] else 0
        )

In [16]:
print(len(X))
print(len(y))
print(set(y))

60
60
{0}


In [17]:
for query in queries:
    print("Query:", query)
    print("Relevant:", relevant_docs[query])

    tokenized_query = query.split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:20]

    retrieved = []

    for idx in top_indices:
        retrieved.append(df.iloc[idx]["doc_id"])

    print("Retrieved:")
    print(retrieved[:10])
    print("-"*50)

Query: lead burden neuropsychology
Relevant: ['NCT00000104']
Retrieved:
['NCT01573013', 'NCT00237952', 'NCT00227409', 'NCT01084694', 'NCT00014898', 'NCT02122289', 'NCT02505425', 'NCT03808506', 'NCT01863199', 'NCT03263091']
--------------------------------------------------
Query: vaccination tetanus immune response
Relevant: ['NCT00000105']
Retrieved:
['NCT00000102', 'NCT04862312', 'NCT04862299', 'NCT04862286', 'NCT04862273', 'NCT04862260', 'NCT04862247', 'NCT04862234', 'NCT04862221', 'NCT04862208']
--------------------------------------------------
Query: hyperthermia body temperature study
Relevant: ['NCT00000106']
Retrieved:
['NCT03296254', 'NCT04498689', 'NCT00003045', 'NCT00766233', 'NCT00002974', 'NCT02409108', 'NCT00876954', 'NCT00796900', 'NCT01904565', 'NCT03436251']
--------------------------------------------------


In [18]:
queries = []

relevant_docs = {}

for i in range(100):
    query = " ".join(df.iloc[i]["cleaned_text"].split()[:5])

    doc_id = df.iloc[i]["doc_id"]

    queries.append(query)

    relevant_docs[query] = [doc_id]

In [19]:
print(len(queries))
print(list(relevant_docs.items())[:3])

100
[('titl congenit adren hyperplasia calcium', ['NCT00000102']), ('titl lead burden alter neuropsycholog', ['NCT00000104']), ('titl vaccin tetanu klh assess', ['NCT00000105'])]


In [20]:
counter = 0

for query in queries:

    tokenized_query = query.split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:20]

    for idx in top_indices:

        counter += 1

        if counter % 50 == 0:
            print("Processed:", counter)

        doc_id = df.iloc[idx]["doc_id"]
        doc_text = df.iloc[idx]["cleaned_text"]

        features = extract_ltr_features(
            query,
            doc_text,
            scores[idx],
            bert_model
        )

        X.append(features)

        y.append(
            1 if doc_id in relevant_docs[query] else 0
        )

Processed: 50
Processed: 100
Processed: 150
Processed: 200
Processed: 250
Processed: 300
Processed: 350
Processed: 400
Processed: 450
Processed: 500
Processed: 550
Processed: 600
Processed: 650
Processed: 700
Processed: 750
Processed: 800
Processed: 850
Processed: 900
Processed: 950
Processed: 1000
Processed: 1050
Processed: 1100
Processed: 1150
Processed: 1200
Processed: 1250
Processed: 1300
Processed: 1350
Processed: 1400
Processed: 1450
Processed: 1500
Processed: 1550
Processed: 1600
Processed: 1650
Processed: 1700
Processed: 1750
Processed: 1800
Processed: 1850
Processed: 1900
Processed: 1950
Processed: 2000


In [21]:
import numpy as np

X = np.array(X)
y = np.array(y)

print(X.shape, y.shape)

(2060, 3) (2060,)


In [22]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [23]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      1.00      0.98       395
           1       0.50      0.06      0.11        17

    accuracy                           0.96       412
   macro avg       0.73      0.53      0.54       412
weighted avg       0.94      0.96      0.94       412



In [24]:
print(len(X))
print(len(y))

2060
2060


In [25]:
import numpy as np

print(np.unique(y, return_counts=True))

(array([0, 1]), array([1976,   84]))


In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [27]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("Training Done")

Training Done


In [28]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.74      0.85       395
           1       0.13      0.88      0.22        17

    accuracy                           0.75       412
   macro avg       0.56      0.81      0.53       412
weighted avg       0.96      0.75      0.82       412



In [29]:
query = "lead burden neuropsychology"

tokenized_query = query.split()
scores = bm25.get_scores(tokenized_query)
top_indices = np.argsort(scores)[::-1][:20]

results = []

for idx in top_indices:

    doc_id = df.iloc[idx]["doc_id"]
    doc_text = df.iloc[idx]["cleaned_text"]

    features = extract_ltr_features(
        query,
        doc_text,
        scores[idx],
        bert_model
    )

    score = model.predict_proba([features])[0][1]

    results.append((doc_id, score))

results = sorted(results, key=lambda x: x[1], reverse=True)

print(results[:10])

[('NCT01573013', np.float64(0.2562013688929041)), ('NCT02778971', np.float64(0.22732076837257734)), ('NCT00237952', np.float64(0.10732074704525651)), ('NCT03808506', np.float64(0.09131946457363585)), ('NCT00013819', np.float64(0.08597181651412768)), ('NCT00388843', np.float64(0.07539856603213858)), ('NCT00014898', np.float64(0.07124553366118413)), ('NCT00227409', np.float64(0.06885681083848359)), ('NCT03099889', np.float64(0.0634044719877095)), ('NCT03645005', np.float64(0.062068989433618736))]
